In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk
from e_1_run_cvae_time_check import train_chunk_time_check
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [512, 256, 128] # [128, 128, 64], [512, 256, 128], [1024. 512, 256], [1024. 512, 256, 128]
batch_size  = 4096 # 1024, 2048, 128-4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk485.pt | 완료 chunks=485
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=485->579 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   486 | epoch    6 chunk   1/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.5322 | KL: 4.7833 | Total: -0.7489
Chunk step   487 | epoch    6 chunk   2/97 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -5.5333 | KL: 4.7836 | Total: -0.7497
Chunk step   488 | epoch    6 chunk   3/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.5283 | KL: 4.8008 | Total: -0.7275
Chunk step   489 | epoch    6 chunk   4/97 | file_idx  96 | BN off    | beta_eff: 1.0000 | Recon: -5.5332 | KL: 4.7967 | Total: -0.7366
Chunk step   490 | epoch    6 chunk   5/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=194->288 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -5.1668 | KL: 4.4364 | Total: -0.7304
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -5.1697 | KL: 4.4503 | Total: -0.7194
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.1679 | KL: 4.4533 | Total: -0.7145
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.1746 | KL: 4.4470 | Total: -0.7276
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.1

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk288.pt | 완료 chunks=288
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=288->291 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   289 | epoch    3 chunk  95/97 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -5.3968 | KL: 4.6634 | Total: -0.7334
Validation @ chunk   289 | Recon: -5.4047 | KL: 4.6649 | Total: -0.7398 | KL_dim: [1.549851, 3.115095]
Chunk step   290 | epoch    3 chunk  96/97 | file_idx   5 | BN off    | beta_eff: 1.0000 | Recon: -5.3949 | KL: 4.6663 | Total: -0.7286
Validation @ chunk   290 | Recon: -5.3940 | KL: 4.6557 | Total: -0.7383 | KL_dim: [1.540994, 3.11465]
Chunk step   291 | epoch    3 chunk  97/97 | file_idx  20 | BN off    | beta_eff: 1.0000 | Recon: -5.3988 | KL: 4.6641 | Total: -0.7347
Validation @ chunk   291 | Recon: -

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=291->385 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.4005 | KL: 4.6627 | Total: -0.7377
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.3960 | KL: 4.6743 | Total: -0.7217
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -5.4016 | KL: 4.6631 | Total: -0.7385
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.4047 | KL: 4.6698 | Total: -0.7349
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN off    | beta_eff: 1.0000 | Recon: -5.4

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk385.pt | 완료 chunks=385
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=385->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.4991 | KL: 4.7486 | Total: -0.7505
Validation @ chunk   386 | Recon: -5.5008 | KL: 4.7556 | Total: -0.7452 | KL_dim: [1.520223, 3.235372]
Chunk step   387 | epoch    4 chunk  96/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -5.4967 | KL: 4.7483 | Total: -0.7484
Validation @ chunk   387 | Recon: -5.4969 | KL: 4.7547 | Total: -0.7423 | KL_dim: [1.512999, 3.241655]
Chunk step   388 | epoch    4 chunk  97/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.4900 | KL: 4.7617 | Total: -0.7283
Validation @ chunk   388 | Recon: 

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk388.pt | 완료 chunks=388
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=388->482 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   389 | epoch    5 chunk   1/97 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -5.4952 | KL: 4.7552 | Total: -0.7400
Chunk step   390 | epoch    5 chunk   2/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -5.4986 | KL: 4.7505 | Total: -0.7481
Chunk step   391 | epoch    5 chunk   3/97 | file_idx  51 | BN off    | beta_eff: 1.0000 | Recon: -5.4957 | KL: 4.7476 | Total: -0.7481
Chunk step   392 | epoch    5 chunk   4/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -5.4985 | KL: 4.7568 | Total: -0.7417
Chunk step   393 | epoch    5 chunk   5/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | Recon: -5.5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk482.pt | 완료 chunks=482
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=482->485 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   483 | epoch    5 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.5385 | KL: 4.7702 | Total: -0.7683
Validation @ chunk   483 | Recon: -5.5288 | KL: 4.7841 | Total: -0.7447 | KL_dim: [1.50193, 3.282208]
Chunk step   484 | epoch    5 chunk  96/97 | file_idx  39 | BN off    | beta_eff: 1.0000 | Recon: -5.5301 | KL: 4.7790 | Total: -0.7510
Validation @ chunk   484 | Recon: -5.5250 | KL: 4.7810 | Total: -0.7440 | KL_dim: [1.499406, 3.281608]
Chunk step   485 | epoch    5 chunk  97/97 | file_idx  42 | BN off    | beta_eff: 1.0000 | Recon: -5.5280 | KL: 4.7931 | Total: -0.7349
Validation @ chunk   485 | Recon: -

In [15]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_512_4096_None_1e-05_1_[15, 24, 78]_chunk485.pt | 완료 chunks=485
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=485->579 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   486 | epoch    6 chunk   1/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.5378 | KL: 4.7887 | Total: -0.7491
Chunk step   487 | epoch    6 chunk   2/97 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -5.5370 | KL: 4.7872 | Total: -0.7499
Chunk step   488 | epoch    6 chunk   3/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.5293 | KL: 4.8015 | Total: -0.7278
Chunk step   489 | epoch    6 chunk   4/97 | file_idx  96 | BN off    | beta_eff: 1.0000 | Recon: -5.5314 | KL: 4.7947 | Total: -0.7367
Chunk step   490 | epoch    6 chunk   5/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2325.pt | 완료 chunks=2325
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=2325->2328 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2326 | epoch   24 chunk  95/97 | file_idx  90 | BN off    | beta_eff: 1.0000 | Recon: -5.8298 | KL: 5.0757 | Total: -0.7541
Validation @ chunk  2326 | Recon: -5.8326 | KL: 5.0851 | Total: -0.7475 | KL_dim: [1.491532, 3.593549]
Chunk step  2327 | epoch   24 chunk  96/97 | file_idx  30 | BN off    | beta_eff: 1.0000 | Recon: -5.8288 | KL: 5.0837 | Total: -0.7451
Validation @ chunk  2327 | Recon: -5.8248 | KL: 5.0785 | Total: -0.7463 | KL_dim: [1.489305, 3.589242]
Chunk step  2328 | epoch   24 chunk  97/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -5.8254 | KL: 5.0898 | Total: -0.7356
Validation @ chunk  

# time check

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [1024, 512, 256] # [128, 128, 64], [256, 256, 128], [512, 256, 128], [1024, 512, 256], [1024, 512, 256, 128]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 1 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_time_check(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
GPU: NVIDIA GeForce RTX 4080 SUPER | cuda capability=(8, 9)
학습 시작 | 이번 실행 chunks=1 | 진행 chunks=0->1 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -0.7828 | KL: 1.1264 | Total: 0.3436
Time | total=68.81s | chunk_load=14.84s | load_wait=14.84s | gpu_move=0.12s | loader_init=0.00s | iter_init=0.00s | batch_fetch=0.00s (0.0000/batch) | h2d=0.00s (0.0000/batch) | fwd=18.93s (0.0012/batch) | bwd=35.73s (0.0022/batch) | clip=6.27s (0.0004/batch) | step=6.20s (0.0004/batch) | loss_item=1.45s | cleanup=0.00s | loop_overhead=0.00s (0.0000/batch) | batches=16384
Validation @ chunk     1 | Recon: -2.1865 | KL: 1.9902 | Total: -0.1963 | KL_dim: [0.360198, 1.630033]
Validation time: 91.65s

=== Time summary for this run ===
chunk_load       

# BN = 5

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN frozen | beta_eff: 1.0000 | Recon: -5.3391 | KL: 4.5933 | Total: -0.7458
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN frozen | beta_eff: 1.0000 | Recon: -5.4229 | KL: 4.6885 | Total: -0.7345
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN frozen | beta_eff: 1.0000 | Recon: -5.4530 | KL: 4.7225 | Total: -0.7305
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN frozen | beta_eff: 1.0000 | Recon: -5.4805 | KL: 4.7367 | Total: -0.7437
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN frozen | beta_eff: 1.0000 | Recon: -5.4988 | KL: 4.7528 | Total: -0.7460
Chunk ste

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_3-0.0001_4-1e-06_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=5 | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN frozen | beta_eff: 1.0000 | Recon: -5.2304 | KL: 4.4934 | Total: -0.7370
Validation @ chunk   384 | Recon: -5.2282 | KL: 4.4857 | Total: -0.7425 | KL_dim: [1.436707, 3.048991]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN frozen | beta_eff: 1.0000 | Recon: -5.2268 | KL: 4.5010 | Total: -0.7258
Validation @ chunk   385 | Recon: -5.2273 | KL: 4.4858 | Total: -0.7415 | KL_dim: [1.42515, 3.060651]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN frozen | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.4922 | Total: -0.7463
Validation @ chunk   386 | Recon: -5.2095 | KL: 4.4750 | Total: -0.7345 | KL_dim: [1.42636

# X,M norm

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001,0.0003, 0.0005
beta        = 0.9
warmup_chunks = None # None or num
num_chunks  = 70
validation_chunk_idxs = [15,24,78]
val_every_chunks = 3
resume_path = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk30.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk100.pt"

In [6]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_norm(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    x_mean=bs_stats["x_mean"],
    x_std=bs_stats["x_std"],
    m_mean=bs_stats["m_mean"],
    m_std=bs_stats["m_std"],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae_xm_norm/bs/cvae_bs_8_128_4096_None_0.001_0.9_None_chunk30.pt | 완료 chunks=30
학습 시작 | 이번 실행 chunks=70 | 진행 chunks=30->100 | files/epoch=100 | bn_chunks=None | warmup_chunks=None
Chunk step    31 | epoch    1 chunk  31/100 | file_idx  30 | BN off    | beta_eff: 0.9000 | Recon: -4.9477 | KL: 5.3827 | Total: -0.1033
Chunk step    32 | epoch    1 chunk  32/100 | file_idx  29 | BN off    | beta_eff: 0.9000 | Recon: -4.8912 | KL: 6.1971 | Total: 0.6862
Chunk step    33 | epoch    1 chunk  33/100 | file_idx  79 | BN off    | beta_eff: 0.9000 | Recon: -4.9538 | KL: 6.2571 | Total: 0.6776
Chunk step    34 | epoch    1 chunk  34/100 | file_idx  44 | BN off    | beta_eff: 0.9000 | Recon: -5.0059 | KL: 5.4327 | Total: -0.1164
Chunk step    35 | epoch    1 chunk  35/100 | file_idx  71 | BN off    | beta_eff: 0.9000 | Recon: -4.9941 | KL: 5.4282 | Total: -0.1088
Chunk step    36 | epoch    1 chunk  36/100 | file_idx  66 | BN off    | beta_eff: 0.9000 | Rec